## Step 1: Load the data and important libraries

In [1]:
import pandas as pd
df = pd.read_csv('hourly_weather_10_days.csv')


In [2]:
## for quick review of data set display 1st 5 rows
df.head()

,timestamp,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
0,2023-03-01 00:00:00,16.6,74.4,5.7,1012.5,9.5
1,2023-03-01 01:00:00,16.2,78.5,5.0,1012.1,10.3
2,2023-03-01 02:00:00,15.3,73.3,4.7,NaN,11.1
3,2023-03-01 03:00:00,15.8,72.4,1.3,1005.0,8.9
4,2023-03-01 04:00:00,20.9,70.6,6.8,1016.3,9.8


## Step 2: Basic Exploration


### Dataframe shape

In [3]:
## show number of rows and columns
df.shape

(240, 6)

### Explore the DataFrame

In [4]:
## show the summary of dataframe, including numbers of null values and datatypes of each column and memory usage
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        240 non-null    object 
 1   temperature_C    228 non-null    float64
 2   humidity_%       224 non-null    float64
 3   wind_speed_kmph  226 non-null    float64
 4   pressure_hPa     223 non-null    float64
 5   visibility_km    228 non-null    float64
dtypes: float64(5), object(1)
memory usage: 11.4+ KB
None


### statisticals details of the dataframe


In [5]:
# count number of non-empty entries, mean, std, min, 1st, 2nd(median) , 3rd quartile and maximum value

df.describe()

,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
count,228.000000,224.000000,226.000000,223.000000,228.000000
mean,21.315789,66.795982,10.105310,1011.884753,9.989474
std,3.421237,8.190300,3.940668,5.187080,1.022166
min,11.500000,47.800000,1.300000,998.100000,6.800000
25%,18.700000,61.075000,6.625000,1008.900000,9.275000
50%,21.900000,66.300000,9.800000,1012.100000,10.000000
75%,23.925000,72.725000,13.500000,1015.100000,10.700000
max,28.700000,88.100000,17.800000,1027.000000,12.600000


## Step: 3 Handle missing values

In [6]:
## count the null values of each column
print(df.isna().sum())

timestamp           0
temperature_C      12
humidity_%         16
wind_speed_kmph    14
pressure_hPa       17
visibility_km      12
dtype: int64


### filling missing values with forward fill, because weather usually changes graduly not instantly

In [7]:
# Fill the missing values in a new DataFrame so that the original DataFrame remains unchanged.
df_filled = df.ffill()

## Step 4: Data Analysis

In [8]:
# convert timesstamp data type (object) to date time
df_filled["timestamp"] = pd.to_datetime(df_filled["timestamp"])

In [9]:
# create a hour column 
df_filled["hour"] = df_filled["timestamp"].dt.hour

### Calculate average daily temperature

In [10]:
daily_avg_temp = df_filled.groupby(df_filled["timestamp"].dt.date).agg(avg_temperature=("temperature_C", "mean"))
print(daily_avg_temp)

            avg_temperature
timestamp                  
2023-03-01        21.550000
2023-03-02        21.366667
2023-03-03        21.433333
2023-03-04        21.554167
2023-03-05        21.670833
2023-03-06        21.858333
2023-03-07        21.037500
2023-03-08        20.762500
2023-03-09        20.945833
2023-03-10        21.716667


### Find max, min, mean for each metric

In [11]:
summary2 = df_filled.drop(columns=["timestamp"]).agg(["mean", "min", "max"])

print(summary2)

      temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km  \
mean      21.389583    66.79875        10.107083   1011.915833      10.002917   
min       11.500000    47.80000         1.300000    998.100000       6.800000   
max       28.700000    88.10000        17.800000   1027.000000      12.600000   

      hour  
mean  11.5  
min    0.0  
max   23.0  


### Which Hour of the Day is the Most Humid on Average?

In [13]:
humidity_by_hour = df_filled.groupby("hour")["humidity_%"].mean()

print(humidity_by_hour)

hour
0     78.17
1     78.42
2     75.89
3     71.94
4     69.31
5     69.34
6     65.77
7     65.05
8     63.49
9     59.65
10    58.71
11    58.91
12    59.53
13    58.33
14    60.54
15    61.05
16    58.71
17    64.03
18    66.70
19    69.19
20    67.46
21    71.43
22    73.75
23    77.80
Name: humidity_%, dtype: float64


In [14]:
most_humid_hour = humidity_by_hour.idxmax()

print("Most humid hour:", most_humid_hour)

Most humid hour: 1


## Step 5: NumPy Matrix Exercises
Convert relevant DataFrame columns into NumPy arrays and perform matrix operations.

In [15]:
# Extract temperature and wind_speed as NumPy arrays
import numpy as np

temp = df_filled['temperature_C'].to_numpy()
wind = df_filled['wind_speed_kmph'].to_numpy()

### a) Reshape into matrix form

In [16]:
# Reshape temperature into a (10, 24) matrix
temp_matrix = temp.reshape(10,24)
print(temp_matrix.shape)

(10, 24)


In [17]:
# Write functions to find min, max, mean across rows
import numpy as np

def row_summary(matrix):
    return {
        "min": np.min(matrix, axis=1),
        "max": np.max(matrix, axis=1),
        "mean": np.mean(matrix, axis=1)
    }

summary = row_summary(temp_matrix)

print("Minimum:", summary["min"])
print("Maximum:", summary["max"])
print("Mean:", summary["mean"])

Minimum: [14.7 15.7 13.6 15.9 12.4 15.5 15.3 13.5 14.3 11.5]
Maximum: [28.2 28.7 25.7 27.1 24.9 26.2 25.9 26.  27.1 28.5]
Mean: [21.55       21.36666667 21.43333333 21.55416667 21.67083333 21.85833333
 21.0375     20.7625     20.94583333 21.71666667]


### b) Normalize the temperature matrix

In [18]:
import numpy as np

# Function to normalize the matrix
def normalize(matrix):
    mean = np.mean(matrix)
    std = np.std(matrix)

    normalized_matrix = (matrix - mean) / std
    return normalized_matrix


# Apply it to temp_matrix
normalized_temp = normalize(temp_matrix)

print(normalized_temp)

[[-1.38980772 -1.50587692 -1.76703261 -1.62194611 -0.14206386 -0.17108116
   0.40926482  0.32221292 -0.05501196  1.97619897  1.97619897  1.2217492
   1.1637146   0.58336862  1.80209518  0.69943781  0.17712643  0.67042051
   0.49631672  0.20614373 -0.92553093 -0.57732335 -0.49027145 -1.9411364 ]
 [-1.59292881 -1.65096341 -1.09963473 -0.40321955  0.17712643 -0.05501196
   0.38024752  0.03203993  0.64140321  2.12128547 -0.25813306  1.0766627
   0.20614373  0.90255891  0.35123022  0.14810913  0.72845511  0.72845511
   0.81550701  0.17712643 -0.43223685 -0.75142714 -1.41882502 -0.98356553]
 [-2.26032669 -1.65096341 -0.83847904 -1.59292881  1.25076649 -0.14206386
  -0.17108116 -0.20009846  0.26417833  1.25076649  0.9896108  -0.05501196
   1.1927319   0.14810913  0.29319563  0.9605935   0.87354161  0.87354161
   0.55435132  0.87354161  0.38024752 -0.14206386 -0.80946174 -1.73801531]
 [-1.36079042 -0.78044444 -1.01258283 -1.56391151  0.03203993  0.20614373
   1.65700868  0.67042051  0.14810913

### c) Apply custom mask/filter

In [19]:
mask = wind > 15
high_wind = wind[mask]

In [20]:
print("Mask:")
print(mask)

print("\nHigh wind readings:")
print(high_wind)

Mask:
[False False False False False False False False False False False  True
  True  True False False False False False False False False False False
 False False False False False False False False False False  True False
 False False False False False False False False False False False False
 False False False False False False False False False  True  True  True
  True False False  True  True False False False False False False False
 False False False False False False False False False False False  True
  True False False False False False False False False False False False
 False False False False False False False False False  True False  True
  True False  True False False False False False False False False False
 False False False False False False False False False False False False
 False  True False False False False False False False False False False
 False False False False False False False False False False  True  True
 False False False False False False False Fa

## Final Challenge: Write Your Own Function
Write a function `daily_summary(matrix)` that takes a NumPy matrix of shape (10, 24) and returns a summary dictionary for each day.

In [21]:
import numpy as np

def daily_summary(matrix):
    summary = {}

    # Loop through each day (each row)
    for day in range(matrix.shape[0]):
        summary[f"Day {day + 1}"] = {
            "min": np.min(matrix[day]),
            "max": np.max(matrix[day]),
            "mean": np.mean(matrix[day])
        }

    return summary


# Call the function
summary = daily_summary(temp_matrix)

# Print the results
for day, values in summary.items():
    print(day, values)

Day 1 {'min': np.float64(14.7), 'max': np.float64(28.2), 'mean': np.float64(21.55)}
Day 2 {'min': np.float64(15.7), 'max': np.float64(28.7), 'mean': np.float64(21.366666666666664)}
Day 3 {'min': np.float64(13.6), 'max': np.float64(25.7), 'mean': np.float64(21.433333333333334)}
Day 4 {'min': np.float64(15.9), 'max': np.float64(27.1), 'mean': np.float64(21.554166666666664)}
Day 5 {'min': np.float64(12.4), 'max': np.float64(24.9), 'mean': np.float64(21.670833333333334)}
Day 6 {'min': np.float64(15.5), 'max': np.float64(26.2), 'mean': np.float64(21.858333333333334)}
Day 7 {'min': np.float64(15.3), 'max': np.float64(25.9), 'mean': np.float64(21.037499999999998)}
Day 8 {'min': np.float64(13.5), 'max': np.float64(26.0), 'mean': np.float64(20.7625)}
Day 9 {'min': np.float64(14.3), 'max': np.float64(27.1), 'mean': np.float64(20.945833333333336)}
Day 10 {'min': np.float64(11.5), 'max': np.float64(28.5), 'mean': np.float64(21.71666666666667)}
